# Fault injector (library)

The real game will not be clean: pieces get knocked, dropped, put on the wrong square, a lamp
is switched on, someone bumps the camera. This notebook turns each fault type in
`gambit_bringup/config/faults.yaml` into concrete world changes (set_pose requests for pieces or
the camera, light_config requests), so `run_fault_tests.ipynb` can check that perception + game
logic **notice** and ask the human instead of acting on a wrong board.

**Deliverable:** robustness tests for the "Gazebo vanishing pieces to test game logic" step of the
Updated plan (and the README's safety goal: never move on a board it is unsure about).

**TODO checklist**
- [x] fault sampling by probability, seeded
- [x] knocked piece, dropped piece, wrong square, lighting change, camera shift -> requests
- [x] expected board after each fault (what the vision *should* report)
- [ ] backend: send the requests (reuse teleport's backends, light_config for lights)
- [ ] `undo(fault)`: restore the clean world after the check
- [ ] decide the pass criteria per fault in run_fault_tests (flagged? correct FEN? no robot move?)

In [ ]:
%run ./data_generator.ipynb

In [ ]:
from dataclasses import dataclass, field

FAULTS_PATH = REPO / "gambit_bringup/config/faults.yaml"
with open(FAULTS_PATH) as f:
    FAULTS = yaml.safe_load(f)

## A fault

`requests` are SetPose (pieces / camera) and `lights` are light dicts for `light_config_command`.
`expected_fen` is the board placement a *perfect* observer would report after the fault (e.g. a
dropped piece is gone, a wrong-square piece is on the wrong square); `None` = unchanged.

In [ ]:
@dataclass
class Fault:
    kind: str
    description: str
    requests: list = field(default_factory=list)
    lights: list = field(default_factory=list)
    expected_fen: str | None = None
    meta: dict = field(default_factory=dict)

def sample_fault_kind(rng, cfg=FAULTS):
    kinds = list(cfg["types"])
    p = np.array([cfg["types"][k]["probability"] for k in kinds], float)
    return str(rng.choice(kinds, p=p / p.sum()))

def _pick_piece(tracker, rng, cfg=FAULTS):
    squares = [sq for sq in tracker.on_square
               if not (cfg.get("protect_kings", True) and tracker.board.piece_type_at(sq) == chess.KING)]
    sq = int(rng.choice(squares))
    return sq, tracker.on_square[sq]

def _uniform(rng, lo_hi):
    return float(rng.uniform(*lo_hi))

def tilt_quat(tilt_deg, heading, rng=None):
    """Quaternion tilting a piece by `tilt_deg` toward `heading` (rad, in the xy plane)."""
    t = math.radians(tilt_deg)
    # tilt axis is horizontal and perpendicular to the heading
    ax, ay = -math.sin(heading), math.cos(heading)
    s = math.sin(t / 2)
    return (ax * s, ay * s, 0.0, math.cos(t / 2))

## Fault generators

Each takes the current `PieceTracker` (the clean position) and an RNG and returns a `Fault`.
They do **not** change the tracker: the tracker stays the ground truth of the *intended* game.

In [ ]:
def knocked_piece(tracker, rng, cfg=FAULTS):
    c = cfg["types"]["knocked_piece"]
    sq, name = _pick_piece(tracker, rng, cfg)
    r, th = _uniform(rng, c["offset_m"]), rng.uniform(0, 2 * math.pi)
    x, y, z = square_center(sq)
    tilt = _uniform(rng, c["tilt_deg"])
    req = SetPose(name, (x + r * math.cos(th), y + r * math.sin(th), z), tilt_quat(tilt, th))
    return Fault("knocked_piece", f"{name} on {chess.square_name(sq)} knocked {r*1000:.1f} mm, tilt {tilt:.0f} deg",
                 [req], meta={"square": chess.square_name(sq), "offset_m": r, "tilt_deg": tilt})

def dropped_piece(tracker, rng, cfg=FAULTS, board=BOARD):
    c = cfg["types"]["dropped_piece"]
    sq, name = _pick_piece(tracker, rng, cfg)
    half = 4 * board["board"]["square"]
    cx, cy = board["board"]["center"]
    if rng.uniform() < c["on_board_probability"]:
        x, y = cx + rng.uniform(-half, half), cy + rng.uniform(-half, half)
        z = board["board"]["thickness"]
    else:   # just outside one side of the playing area, on the frame or the table
        side = rng.integers(4); d = half + _uniform(rng, c["off_board_margin_m"]); u = rng.uniform(-half, half)
        x, y = [(cx + d, cy + u), (cx - d, cy + u), (cx + u, cy + d), (cx + u, cy - d)][side]
        z = board["board"]["thickness"] if max(abs(x - cx), abs(y - cy)) < board_outer_size(board) / 2 else 0.0
    z += piece_radius(board)                                   # lying on its side
    heading = rng.uniform(0, 2 * math.pi)
    req = SetPose(name, (x, y, z), tilt_quat(90.0, heading))
    b = tracker.board.copy(); b.remove_piece_at(sq)            # a perfect observer sees it missing
    return Fault("dropped_piece", f"{name} from {chess.square_name(sq)} dropped at ({x:.3f}, {y:.3f})",
                 [req], expected_fen=b.board_fen(), meta={"square": chess.square_name(sq)})

def wrong_square(tracker, rng, cfg=FAULTS):
    """Move a piece to a neighbouring square (as if the last move landed one square off)."""
    c = cfg["types"]["wrong_square"]
    sq, name = _pick_piece(tracker, rng, cfg)
    f, r, k = chess.square_file(sq), chess.square_rank(sq), c["max_distance_squares"]
    cands = [chess.square(f + df, r + dr) for df in range(-k, k + 1) for dr in range(-k, k + 1)
             if (df or dr) and 0 <= f + df < 8 and 0 <= r + dr < 8]
    if not c["allow_occupied"]:
        cands = [s for s in cands if s not in tracker.on_square]
    if not cands:
        return knocked_piece(tracker, rng, cfg)               # boxed in: degrade to a knock
    to = int(rng.choice(cands))
    b = tracker.board.copy(); b.set_piece_at(to, b.remove_piece_at(sq))
    return Fault("wrong_square", f"{name} {chess.square_name(sq)} -> {chess.square_name(to)}",
                 [SetPose(name, square_center(to))], expected_fen=b.board_fen(),
                 meta={"from": chess.square_name(sq), "to": chess.square_name(to)})

def lighting_change(tracker, rng, cfg=FAULTS):
    c = cfg["types"]["lighting_change"]
    s = _uniform(rng, c["intensity_scale"])
    light = {"diffuse": (min(0.85 * s, 1.0),) * 3,
             "direction": tuple(float(v) for v in _rotate_dir((-0.3, -0.2, -1.0), c["direction_jitter_deg"], rng)),
             "cast_shadows": True}
    return Fault("lighting_change", f"sun x{s:.2f}", lights=[light], meta={"scale": s})

def camera_shift(tracker, rng, cfg=FAULTS, board=BOARD):
    c = cfg["types"]["camera_shift"]
    x, y, z, roll, pitch, yaw = camera_pose(board)
    d = rng.normal(size=3); d *= _uniform(rng, c["position_m"]) / np.linalg.norm(d)
    rot = np.radians(rng.normal(size=3)); rot *= math.radians(_uniform(rng, c["rotation_deg"])) / np.linalg.norm(rot)
    pose = (x + d[0], y + d[1], z + d[2], roll + rot[0], pitch + rot[1], yaw + rot[2])
    return Fault("camera_shift", f"camera moved {np.linalg.norm(d)*1000:.1f} mm",
                 [camera_set_pose(pose)], meta={"pose": pose})

GENERATORS = {"knocked_piece": knocked_piece, "dropped_piece": dropped_piece, "wrong_square": wrong_square,
              "lighting_change": lighting_change, "camera_shift": camera_shift}

def sample_fault(tracker, rng, cfg=FAULTS):
    return GENERATORS[sample_fault_kind(rng, cfg)](tracker, rng, cfg)

def fault_schedule(n_plies, rng, cfg=FAULTS):
    """Plies after which to inject a fault: ~faults_per_game, at least min_plies_between apart."""
    out, last = [], -cfg["min_plies_between"]
    p = cfg["faults_per_game"] / max(n_plies, 1)
    for i in range(1, n_plies + 1):
        if i - last >= cfg["min_plies_between"] and rng.uniform() < p:
            out.append(i); last = i
    return out

## Applying and undoing (TODO)

In [ ]:
def inject(fault, backend):
    """TODO: backend.send_all(fault.requests); for each light run light_config_command (gz CLI)
    or the ROS bridge. Wait capture.settle_s before grabbing a frame."""
    raise NotImplementedError("TODO: inject")

def undo(fault, tracker, backend):
    """TODO: restore the clean world: tracker.requests_for_all() for pieces, camera_set_pose(
    camera_pose()) for the camera, and the default sun from chess_world.sdf for lights."""
    raise NotImplementedError("TODO: undo")

## Self-check

In [ ]:
rng = np.random.default_rng(FAULTS["seed"])
t = PieceTracker.from_fen()
for m in ["e2e4", "e7e5", "g1f3"]:
    t.apply_move(m)
clean = t.board.board_fen()

counts = {k: 0 for k in GENERATORS}
for _ in range(200):
    f = sample_fault(t, rng)
    counts[f.kind] += 1
    assert f.requests or f.lights
    if f.kind == "knocked_piece":
        lo, hi = FAULTS["types"]["knocked_piece"]["offset_m"]
        sq = chess.parse_square(f.meta["square"])
        off = math.dist(f.requests[0].position[:2], square_center(sq)[:2])
        assert lo - 1e-9 <= off <= hi + 1e-9
    if f.kind in ("knocked_piece", "dropped_piece", "wrong_square"):
        assert "K_" not in f.requests[0].name          # kings protected
    if f.expected_fen:
        assert f.expected_fen != clean
    q = f.requests[0].orientation if f.requests else (0, 0, 0, 1)
    assert abs(sum(v * v for v in q) - 1) < 1e-9
assert t.board.board_fen() == clean and t.check_consistent()     # tracker untouched
assert all(v > 0 for v in counts.values()), counts

w = wrong_square(t, np.random.default_rng(1))
assert chess.square_distance(chess.parse_square(w.meta["from"]), chess.parse_square(w.meta["to"])) == 1
s = fault_schedule(80, np.random.default_rng(0))
assert all(b - a >= FAULTS["min_plies_between"] for a, b in zip(s, s[1:]))
print("fault_injector self-check OK", counts)